## Load BLIP-2 

Loads the Salesforce BLIP-2 Flan-T5-XL processor and pretrained model for inference. This notebook also serves as the reusable loader for other .ipynb notebooks.

In [1]:
import os
from pathlib import Path

_current_dir = Path.cwd().resolve()
PROJECT_ROOT = _current_dir.parent if _current_dir.name.lower() == "scripts" else _current_dir

MODEL_ID = "Salesforce/blip2-flan-t5-xl"
HF_CACHE = PROJECT_ROOT / "hf_cache"
OFFLOAD_DIR = PROJECT_ROOT / "offload"
HF_CACHE.mkdir(parents=True, exist_ok=True)
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Keep downloaded files in one shared project cache.
os.environ["HF_HOME"] = str(HF_CACHE)

The first run loads the model; subsequent runs in the same kernel reuse the existing objects. This does not preserve in-memory weights across kernel restarts.

In [2]:
import torch
from transformers import Blip2ForConditionalGeneration, Blip2Processor

# in notebooks sharing one kernel, these objects already exist after first load.
if "processor" not in globals() or processor is None:
    processor = Blip2Processor.from_pretrained(
        MODEL_ID,
        use_fast=False,
        cache_dir=str(HF_CACHE / "hub"),
    )

if "model" not in globals() or model is None:
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("This configuration requires CUDA with BF16 support.")

    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        use_safetensors=True,
        device_map="auto",
        max_memory={0: "12GiB"},
        low_cpu_mem_usage=True,
        offload_folder=str(OFFLOAD_DIR),
        cache_dir=str(HF_CACHE / "hub"),
    )

model.eval()

# keep the processor consistent with the currently loaded query tensor.
processor.num_query_tokens = int(model.query_tokens.shape[1])

# fail early if an incompatible tokenizer/model combination is loaded.
image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
if image_token_id != model.config.image_token_index:
    raise ValueError("Processor and model use different <image> token IDs.")

print(f"BLIP-2 ready ({processor.num_query_tokens} query tokens).")


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

C:\Users\antho\miniconda3\envs\blip2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\antho\Documents\BLIP2_STATS425_Final\hf_cache\hub\models--Salesforce--blip2-flan-t5-xl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (7 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/9.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

BLIP-2 ready (32 query tokens).


In [3]:
# test to confirm that all model weights loaded properly 

print("Model architecture:")
print(model.config.architectures)

print("\nNumber of pretrained queries:")
print(model.config.num_query_tokens)

print("\nQuery embedding shape:")
print(tuple(model.query_tokens.shape))

print("\nQ-Former hidden dimension:")
print(model.config.qformer_config.hidden_size)

print("\nVision encoder hidden dimension:")
print(model.config.vision_config.hidden_size)

print("\nVision encoder layers:")
print(model.config.vision_config.num_hidden_layers)

Model architecture:
['Blip2ForConditionalGeneration']

Number of pretrained queries:
32

Query embedding shape:
(1, 32, 768)

Q-Former hidden dimension:
768

Vision encoder hidden dimension:
1408

Vision encoder layers:
39
